In [ ]:
import pandas as pd


df = pd.read_csv('/mnt/data/cc444af3-2e3f-46de-86be-d98536374d81.csv')
df.head()

In [ ]:

# =============================
# DATA CLEANING & VALIDATION
# =============================

# 1. Convert joining_date
df['joining_date_clean'] = pd.to_datetime(df['joining_date'], errors='coerce')
failed_joining_date = df['joining_date_clean'].isna().sum()

# 2. Clean employee_id
df['employee_id_clean'] = df['employee_id'].astype(str).str.strip().str.upper()
duplicate_employees = df['employee_id_clean'].duplicated().sum()

# 3. Standardize department & salary
df['department_clean'] = df['department'].str.strip().str.title()
df['salary_clean'] = pd.to_numeric(df['salary'], errors='coerce')
avg_salary_dept = df.groupby('department_clean')['salary_clean'].mean()

# 4. Clean age
df['age_clean'] = pd.to_numeric(df['age'], errors='coerce')
invalid_age_valid_salary = df[(df['salary_clean'].notna()) & (df['age_clean'].isna())]

# 5. Salary outliers (IQR)
Q1 = df['salary_clean'].quantile(0.25)
Q3 = df['salary_clean'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
salary_outliers = df[(df['salary_clean'] < lower) | (df['salary_clean'] > upper)]

# 6. Performance rating median
df['performance_rating_clean'] = pd.to_numeric(df['performance_rating'], errors='coerce')
df['designation_clean'] = df['designation'].str.strip().str.title()
median_rating = df.groupby('designation_clean')['performance_rating_clean'].median()

# 7. Invalid promotion dates
df['last_promotion_date_clean'] = pd.to_datetime(df['last_promotion_date'], errors='coerce')
invalid_promotion = df[df['last_promotion_date_clean'] < df['joining_date_clean']]

# 8. Experience mismatch
df['experience_clean'] = pd.to_numeric(df['experience_years'], errors='coerce')
experience_age_mismatch = df[df['experience_clean'] > df['age_clean']]

# 9. Active employees count
df['is_active_clean'] = df['is_active'].astype(str).str.lower().map({'true':True,'false':False})
active_count = df[df['is_active_clean'] == True].groupby('designation_clean').size()

# 10. Inactive with recent promotions
recent_cutoff = pd.Timestamp.today() - pd.DateOffset(years=2)
inactive_recent_promo = df[(df['is_active_clean'] == False) & 
                           (df['last_promotion_date_clean'] >= recent_cutoff)]

# 11. Tenure calculation
df['tenure_years'] = (pd.Timestamp.today() - df['joining_date_clean']).dt.days / 365
p90 = df['tenure_years'].quantile(0.9)
high_tenure = df[df['tenure_years'] > p90]

# 12. Dept >25% missing salary
salary_missing_pct = df.groupby('department_clean')['salary_clean'].apply(lambda x: x.isna().mean())
dept_high_missing = salary_missing_pct[salary_missing_pct > 0.25]

# 13. High performance but low salary
median_salary = df['salary_clean'].median()
df['high_perf_low_salary_flag'] = np.where(
    (df['performance_rating_clean'] >= 4) & 
    (df['salary_clean'] < median_salary), 1, 0)

# 14. No promotion but >5 years exp
no_promo_high_exp = df[(df['last_promotion_date_clean'].isna()) & 
                       (df['experience_clean'] > 5)]

# 15. Multi-constraint violation
df['constraint1'] = df['joining_date_clean'].isna()
df['constraint2'] = df['salary_clean'].isna()
df['constraint3'] = df['age_clean'].isna()
df['violation_count'] = df[['constraint1','constraint2','constraint3']].sum(axis=1)
df['multi_violation_flag'] = df['violation_count'] >= 2

# Additional Advanced Questions

df['promotion_gap'] = (df['last_promotion_date_clean'] - 
                       df['joining_date_clean']).dt.days / 365

dept_avg_salary = df.groupby('department_clean')['salary_clean'].transform('mean')
dept_median_perf = df.groupby('department_clean')['performance_rating_clean'].transform('median')

df['salary_exp_ratio'] = df['salary_clean'] / df['experience_clean']
p95 = df['salary_exp_ratio'].quantile(0.95)

attrition_rate = df.groupby('department_clean')['is_active_clean'].apply(lambda x: (~x).mean())

df['valid_fields'] = df[['joining_date_clean','salary_clean','age_clean','experience_clean']].notna().sum(axis=1)
df['quality_score'] = df['valid_fields'] / 4
df['low_quality_flag'] = df['quality_score'] < 0.75

print("Notebook execution completed successfully.")
